In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


In [13]:
df = pd.read_csv("data/energy_firms.csv")

# Question 1 - Explore electricity use before and after consulting

We have found that there are an equal number of firms treated with the consulting and those not treated with the consulting (240 each condition). Additionally we have found that there are an even number of firms within the textiles and chemicals sectors within these groups (120 of each sector within the treated and 120 of each sector within the control). Additionally the prices of wages, energy, and output have been confirmed to be constant per plant.

Randomization:
When analyzing output, labor, and electricity from the treated and control plants from the pretreatment monts (-3 to 0) textiles was found to be well-balanced (p-values 0.64 to 0.99). However in Chemicals the treated plants are significantly larger on all three measures (p = 0.002 to 0.019), which causes a potential imbalance in the overall control and treatment sample. Because this is liekly due to chance of random draw, analysis will be conducted by sector as well as combined. 


In [14]:
print(df.head())
print(df.describe())


  firm_id    sector  treated  month  adjust_month  labor_adjustable  \
0    F001  Textiles        0     -3             4                 1   
1    F001  Textiles        0     -2             4                 1   
2    F001  Textiles        0     -1             4                 1   
3    F001  Textiles        0      0             4                 1   
4    F001  Textiles        0      1             4                 0   

       output       labor  electricity  output_price         wage  \
0  100.239636  174.153124  4184.171250    7199.02915  1163.478189   
1   99.011563  172.124961  4411.388868    7199.02915  1163.478189   
2  102.154783  170.255851  4276.845098    7199.02915  1163.478189   
3   99.568497  172.706398  4446.593229    7199.02915  1163.478189   
4   99.699985  174.145341  4281.219976    7199.02915  1163.478189   

   energy_price  
0     36.693608  
1     36.693608  
2     36.693608  
3     36.693608  
4     36.693608  
           treated        month  adjust_month  lab

In [15]:
cols = ['firm_id', 'sector', 'treated', 'month', 'adjust_month', 'labor_adjustable',
 'output', 'labor', 'electricity', 'output_price', 'wage', 'energy_price']

nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.any() else "No NaNs found.")



No NaNs found.


In [16]:
print(df.groupby("treated")["firm_id"].nunique())

print(df.groupby(["sector", "treated"])["firm_id"].nunique())

treated
0    240
1    240
Name: firm_id, dtype: int64
sector     treated
Chemicals  0          120
           1          120
Textiles   0          120
           1          120
Name: firm_id, dtype: int64


In [19]:
price_vars = ["output_price", "wage", "energy_price"]

# Number of distinct values per plant: should be 1 for every plant
print(df.groupby("firm_id")[price_vars].nunique().max())

output_price    1
wage            1
energy_price    1
dtype: int64


In [17]:
pre = df[df["month"] <= 0]

# Overall
print(pre.groupby("treated")[["output", "labor", "electricity"]].mean())

# By sector
print(pre.groupby(["sector", "treated"])[["output", "labor", "electricity"]].mean())

            output       labor  electricity
treated                                    
0        76.595411  161.888360  4108.707907
1        80.871133  174.581118  4413.708307
                      output       labor  electricity
sector    treated                                    
Chemicals 0        75.426559  153.384424  3892.377016
          1        82.442348  179.507304  4498.675985
Textiles  0        77.764263  170.392296  4325.038798
          1        79.299919  169.654932  4328.740629


In [18]:
outcomes = ["output", "labor", "electricity"]

pre = df[df["month"] <= 0]
plant_pre = pre.groupby(["firm_id", "sector", "treated"])[outcomes].mean().reset_index()

def balance(d):
    rows = {}
    for v in outcomes:
        t = d.loc[d["treated"] == 1, v]
        c = d.loc[d["treated"] == 0, v]
        rows[v] = {
            "control_mean": c.mean(),
            "treated_mean": t.mean(),
            "diff": t.mean() - c.mean(),
            "std_diff": (t.mean() - c.mean()) / np.sqrt((t.var() + c.var()) / 2),
            "p_value": stats.ttest_ind(t, c, equal_var=False).pvalue,
        }
    return pd.DataFrame(rows).T

print("All plants");  print(balance(plant_pre).round(3))
for s, d in plant_pre.groupby("sector"):
    print(s);         print(balance(d).round(3))

All plants
             control_mean  treated_mean     diff  std_diff  p_value
output             76.595        80.871    4.276     0.177    0.053
labor             161.888       174.581   12.693     0.186    0.042
electricity      4108.708      4413.708  305.000     0.149    0.103
Chemicals
             control_mean  treated_mean     diff  std_diff  p_value
output             75.427        82.442    7.016     0.304    0.019
labor             153.384       179.507   26.123     0.409    0.002
electricity      3892.377      4498.676  606.299     0.317    0.015
Textiles
             control_mean  treated_mean   diff  std_diff  p_value
output             77.764        79.300  1.536     0.061    0.637
labor             170.392       169.655 -0.737    -0.010    0.937
electricity      4325.039      4328.741  3.702     0.002    0.989
